# 4. Generation

The causal models (HyenaDNA, Evo 2, GENERator) predict the next token from the previous ones,
so they can continue a DNA prompt. `generate_sequence` takes

- `prompt`, `checkpoint`, `max_new_tokens`,
- `temperature` (0 = greedy, always the most likely token; higher = more random),
- `top_k` (sample only from the k most likely tokens; the default means no filtering),
- `seed` (makes sampling reproducible),

and returns `generated_sequence` (new bases only), `full_sequence`, `num_new_tokens` and
`num_new_bases`.

> Run the series in order the first time: notebook 01 explains the setup. Every notebook
> starts by connecting to the services through `tutorial_utils.py` next to it; a service
> that isn't running is skipped.

In [1]:
from tutorial_utils import HBB, connect, default_checkpoints

clients = connect()
causal = {
    name: c
    for name, c in clients.items()
    if "generate_sequence" in {t["name"] for t in c.list_tools()}
}
defaults = default_checkpoints(causal)
print("causal models:", defaults)

server http://localhost | token set: True | up: ntv3, ntv2, dnabert2, hyenadna, evo2, generator, genalm, grover


causal models: {'hyenadna': 'medium-160k', 'evo2': '7b', 'generator': 'v2-eukaryote-1.2b'}


## Continue HBB

We give each model the first 60 bp of HBB and compare its continuation with the real next
bases. For GENERator a token is a 6-mer, so `max_new_tokens=5` gives 30 bases; for the
others a token is one base.

In [2]:
prompt = HBB[:60]
print(f"{'real HBB':<10} {HBB[60:90]}")
for name, c in causal.items():
    tokens = 5 if name == "generator" else 30
    out = c.call(
        "generate_sequence", prompt=prompt, checkpoint=defaults[name], max_new_tokens=tokens, seed=1
    )
    same = sum(a == b for a, b in zip(out["generated_sequence"], HBB[60:90]))
    print(
        f"{name:<10} {out['generated_sequence'][:30]}  ({out['num_new_bases']} bases, {same}/30 match the real sequence)"
    )

real HBB   GTGGATGAAGTTGGTGGTGAGGCCCTGGGC


hyenadna   ACTGCTGGTGACTTTGGGACAGCAAGGCCC  (30 bases, 11/30 match the real sequence)


evo2       GCTGAGGATGACGGTGTCGAAGCTCTGAGC  (30 bases, 19/30 match the real sequence)
generator  GCGGAAGACGTGGGCGCCGATGCGCTGGCC  (30 bases, 20/30 match the real sequence)


## Temperature

Greedy decoding (`temperature=0`) always picks the most likely base, so it is
deterministic and, over longer outputs, tends to fall into repeats; sampling at a moderate
temperature gives more varied sequence. Compare the outputs and their GC content:

In [3]:
name = "evo2" if "evo2" in causal else next(iter(causal))
for temperature in [0.0, 0.7, 1.0]:
    out = causal[name].call(
        "generate_sequence",
        prompt=prompt,
        checkpoint=defaults[name],
        max_new_tokens=60,
        temperature=temperature,
        seed=1,
    )
    seq = out["generated_sequence"]
    gc = (seq.count("G") + seq.count("C")) / len(seq)
    print(f"{name} T={temperature}: {seq}  GC={gc:.2f}")

evo2 T=0.0: GTGGAAGACGACGGTGTGGATGAGCTGGCCAAGCTGAAGGTGGTGCTGCACTACCAGCTG  GC=0.60


evo2 T=0.7: GTTGAGGATGACGGTGTTGAAGAGCTGTCCAGGTGAGATTCTCAATCCAGCTCAACAAGG  GC=0.50


evo2 T=1.0: GCTGAGGATGACGGTGTCGAAGCTCTGAGCAGGTTAGATTCTCTCTCCCTCTCTCCAAGG  GC=0.55


## GENERator and lengths that aren't a multiple of 6

GENERator tokenizes DNA into 6-mers, so by default it rejects prompts (and embedding inputs)
whose length isn't a multiple of 6. `remainder="trim_left"` drops the first `len % 6` bases
(the authors' recipe) and `remainder="pad_left"` pads; the response says how many bases were
adjusted.

In [4]:
if "generator" in causal:
    g = causal["generator"]
    try:
        g.call("generate_sequence", prompt=HBB[:62], max_new_tokens=2)
    except RuntimeError as exc:
        print("default:", exc)
    out = g.call(
        "generate_sequence", prompt=HBB[:62], max_new_tokens=2, remainder="trim_left", seed=1
    )
    print(
        "trim_left:",
        {
            k: out[k]
            for k in ["remainder_bases", "num_new_tokens", "num_new_bases", "generated_sequence"]
            if k in out
        },
    )

default: Error executing tool generate_sequence: GENERator reads DNA as non-overlapping 6-mers, and this sequence is 62 bp (62 % 6 = 2). Pass a length that is a multiple of 6, or set remainder='trim_left' (drop the first 2 bases, as the GENERator authors do) or remainder='pad_left' (prepend 4 'A' bases). Bases are never dropped silently.
trim_left: {'remainder_bases': 2, 'num_new_tokens': 2, 'num_new_bases': 12, 'generated_sequence': 'GCACCCGAACGG'}


**Next:** [5. NTv3 species models](05_ntv3_annotation.ipynb)